In [2]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper, PDFTableExtractor

helper = Helper()

In [ ]:
def export_to_excel(results, output_path="output.xlsx"):
    """
    Converts extraction results into an Excel file.

    PARAMETERS:
    - results: dict → {page_no: [DataFrames]}
    - output_path: str → output file path

    LOGIC:
    - Each page becomes a separate sheet
    - Multiple tables are stacked vertically with spacing
    """

    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:

        for page_no, dfs in results.items():

            sheet_name = f"Page_{page_no}"
            start_row = 0

            for idx, df in enumerate(dfs):

                if df.empty:
                    continue

                # Write table
                df.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    startrow=start_row,
                    index=False,
                    header=False
                )

                # Move down for next table (add spacing)
                start_row += len(df) + 3  # 3-row gap

    print(f"Excel saved to: {output_path}")
    
    
    


In [20]:
path = "SHRI.pdf"
extractor = PDFTableExtractor(path)

result = extractor.extract(
    page_numbers=[5,6,7,8,9,10,11,12,13,14],
    bboxes=[(428, 110, 620, 800)],
    method="other",
    x_thresh=0.1
)

export_to_excel(result,"output_.xlsx")

Excel saved to: output_.xlsx


In [3]:
pdf_path = r"C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF.pdf"
pdf_path = r"TEXT_PDF.pdf"

In [ ]:
# ==============================
# 1. EXTRACT TEXT
# ==============================
helper = Helper()
text_data = helper.get_pdf_text(pdf_path)
print("TEXT SAMPLE:", text_data[0][:10])


In [ ]:
# ==============================
# 2. GET ALL BLOCKS + IMAGES
# ==============================
helper = Helper()
all_data = helper.get_all_pdf_data(pdf_path)
pprint.pprint(all_data[0])
# print(all_data[0])

In [ ]:
# ==============================
# 3. CLIPPED DATA (EDIT BBOX)
# ==============================
helper = Helper()
bboxes = [
    (0, 0, 300, 400),
    (100, 200, 400, 600)
]

try:
    clipped = helper.get_clipped_data(pdf_path, bboxes)
    print("CLIPPED SAMPLE:", clipped[0])
except Exception as e:
    print("Clipping skipped:", e)


In [ ]:
# ==============================
# 4. DRAW LINES + RECTS
# ==============================
lines = [
    ((50, 50), (300, 50)),
    ((100, 100), (400, 100))
]
rects = [(50, 50, 200, 300)]
pages = [1]

pdf_path = r"STR.pdf"
output_draw = pdf_path.replace(".pdf", "_drawn.pdf")
helper.draw_lines_on_pdf( pdf_path,lines,rects,pages,output_draw)



Modified PDF saved to: C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF_drawn.pdf


In [5]:
# ==============================
# 5. LINE BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"STR_bbox_mask.pdf"
output_path =helper.draw_boundaries_on_lines(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['STR_bbox_mask_line_hltd.pdf']>

In [5]:
# ==============================
# 6. BLOCK BOUNDARIES
# ==============================
helper = Helper()
output_path = helper.draw_boundaries_on_pdf(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['SAMPLE_block_highlighted.pdf']>

In [3]:
# ==============================
# 7. SPAN BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"KOT_bbox_mask.pdf"
output_path = helper.draw_span_boundaries(pdf_path)
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['KOT_bbox_mask_span_hltd.pdf']>

In [4]:
# ==============================
# 8. BBOX ONLY TEXT
# ==============================
helper = Helper()
pdf_path = r"STR.pdf"
output_path = helper.mask_outside_bboxes(pdf_path,[(4.1, 116.3, 268.64, 724.84)])
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['STR_bbox_mask.pdf']>

In [ ]:
# ==============================
# 9. SINGLE BBOX DRAW
# ==============================
bbox = (100, 100, 400, 400)
helper = Helper()
helper.draw_bboxes_on_pdf(pdf_path, bbox)



In [ ]:
lines = [
    ((110, 0), (110, 812)),# Vertical line
    ((0, 350), (812, 350)),
    ((570, 0), (570, 812))
]
pages = [12, 14,16]
bboxes = [[0, 120, 180, 812],[180, 85, 360, 812]] #[(0, 85, 180, 812),(180, 85, 360, 812),(0,100,270,812),(0,100,350,812)]
pages = [i for i in range(1,110)]
sample_path = ""
Helper.draw_lines_on_pdf(sample_path, lines, bboxes, pages, dry_path)

In [ ]:
import fitz
import pytesseract
from PIL import Image
import io, re

def get_proper_fund_names(path: str):
    title = {}
    pattern ="((?:LI?i?C|BSE|BANK|SMALL|HEALTH|MNEY|[aA]n\\s*open).*?(?:FUND|Path|ETF|FTF|EOF|FOF|PLAN|SAVER|tax saving scheme|small cap stocks)\\s*(?:FUND\\s*OF\\s*FUND)?)"
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            clip = fitz.Rect(300, 0, 595, 80)
            pix = page.get_pixmap(clip=clip, dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes()))
            text = pytesseract.image_to_string(img)
            cleaned = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            if matches := re.findall(pattern, cleaned, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _])
                print(f"{pgn}:matched {matches[0]}")
            # print(f"[OCR] Page {pgn}: {cleaned}")
            # if cleaned:
            #     title[pgn] = cleaned
    return title
path = r"C:\Users\kaustubh.keny\OneDrive - Cogencis Information Services Ltd\Documents\MUTUAL FUND FACTSHEET FY19-25\2021_changed\LIC Mutual Fund\25_31-Dec-21_FS.pdf"
title = get_proper_fund_names(path)

In [2]:
import re, fitz

def get_proper_fund_names(path: str, pattern:str,clip:tuple):
    title = {} 
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            text = " ".join(page.get_text("text", clip = clip).split("\n")) #clip = (0, 0, 210, 155)
            text = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            # print(f"{pgn}:-{text}")
            if matches := re.findall(pattern, text, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _ ])
                print(pgn,matches[0])
    return title
path = r"74_30-Apr-26_IF.pdf"
pattern =  ".+?FUND"
clip = (40, 0, 535, 75)
# pattern = "(MIRAE.*?)NSE\\s*[Ss]ymbol"
title = get_proper_fund_names(path,pattern,clip)

8 ENHANCED EQUITY FUND
10 ACCELERATOR FUND
12 PENSION ENHANCED EQUITY FUND
14 PENSION INDEX FUND
15 GROUP EQUITY FUND
16 BLUE CHIP EQUITY FUND
18 OPPORTUNITY FUND
20 DEBT FUND
21 PENSION DEBT FUND
22 SECURE FUND
23 PENSION SECURE FUND
24 CONSERVATIVE FUND
26 BALANCED FUND
28 PENSION BALANCED FUND
30 STABLE FUND
32 FLEXI CAP FUND
34 PENSION NIFTY 500 MOMENTUM  QUALITY 50 INDEX FUND
36 LIQUID FUND
37 MIDCAP FUND


In [ ]:
def extract_clipped_data(input:str, pages:list, bboxes:list):
        
        document = fitz.open(input)
        final_list = []
    
        for pgn in pages:
            page = document[pgn]
            
            all_blocks = [] #store every data from bboxes
            
            for bbox in bboxes:
                blocks, seen_blocks = [], set()  #store unique blocks based on content and bbox
                
                page_blocks = page.get_text('dict', clip=bbox)['blocks']
                for block in page_blocks:
                    if block['type'] == 0 and 'lines' in block: #type 0 means text block
                        #hash_key
                        block_key = (tuple(block['bbox']), tuple(tuple(line['spans'][0]['text'] for line in block['lines'])))
                        if block_key not in seen_blocks:
                            seen_blocks.add(block_key)
                            blocks.append(block)

                sorted_blocks = sorted(blocks, key=lambda x: (x['bbox'][1], x['bbox'][0]))
                all_blocks.append(sorted_blocks)

            final_list.append({
                "pgn": pgn,
                "block": all_blocks #will be list[list,list,..]
            })

        document.close()
        return final_list
    
def extract_data_relative_line(path: str, line_x: float, side: str):
    doc = fitz.open(path)
    pages = doc.page_count

    final_list = []

    for pgn in range(pages):
        page = doc[pgn]

        blocks = page.get_text("dict")["blocks"]
        sorted_blocks = sorted(blocks, key=lambda x: (x["bbox"][1], x["bbox"][0]))
        extracted_blocks = []

        # Keep track of blocks to avoid duplicates
        added_blocks = set()

        for block in sorted_blocks:
            block_id = id(block)  # Unique identifier for the block

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    origin = span["origin"]
                    x0, _ = origin

                    # Check the side condition
                    if side == "left" and x0 < line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added
                    elif side == "right" and x0 > line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added

      
        final_list.append({
            "pgn": pgn,
            "blocks": extracted_blocks
        })

    doc.close()

    return final_list
  
def get_clipped_data(input:str, bboxes:list[set], *args):
    
        document = fitz.open(input)
        final_list = []
        if args:
            pages = list(args)
        else:
            pages = [i for i in document.page_count]
        
        for pgn in pages:
            page = document[pgn]

            blocks = []
            for bbox in bboxes:
                blocks.extend(page.get_text('dict', clip = bbox)['blocks']) #get all blocks
            
            filtered_blocks = [block for block in blocks if block['type']== 0 and 'lines' in block]
            # sorted_blocks = sorted(filtered_blocks, key= lambda x: (x['bbox'][1], x['bbox'][0]))
             # Extract text from sorted blocks
            extracted_text = []
            for block in filtered_blocks:
                block_text = []
                for line in block['lines']:
                    line_text = " ".join(span['text'] for span in line['spans'])
                    block_text.append(line_text)
                extracted_text.append("\n".join(block_text))
            
            final_list.append({
            "pgn": pgn,
            "block": filtered_blocks,
            "text": extracted_text
            })
            
            
        document.close()
        return final_list
    
def extract_clipped_text_all_pages(pdf_path, clip_coords):
    results = {}
    doc = fitz.open(pdf_path)
    clip_rect = fitz.Rect(*clip_coords)
    try:
        for page_number, page in enumerate(doc):
            text = page.get_text("text", clip=clip_rect).strip()
            results[page_number] = text
    finally:
        doc.close()
    return results

In [21]:
from openpyxl import Workbook
from openpyxl.styles import Font
from datetime import datetime, timedelta

def last_day_previous_month():
    today = datetime.today()
    first_day_this_month = today.replace(day=1)
    last_day_prev_month = first_day_this_month - timedelta(days=1)
    return last_day_prev_month.strftime("%d %b %Y")

def create_excel_blueprint(fund_list, output_file="mutual_funds_blueprint.xlsx"):
    wb = Workbook()

    # ---------------------------
    # 1. INDEX SHEET
    # ---------------------------
    ws_index = wb.active
    ws_index.title = "Index"
    website = "www.google.com"

    # Headers
    ws_index["A1"] = "SCHEMES"
    ws_index["C1"] = "Fund House Details"

    ws_index["A1"].font = Font(bold=True)
    ws_index["C1"].font = Font(bold=True)

    # Fund list with WORKING hyperlinks (cross-platform)
    for i, fund in enumerate(fund_list, start=2):
        sheet_name = fund[:31].replace('"', '')  # sanitize quotes

        cell = ws_index[f"A{i}"]
        cell.value = f'=HYPERLINK("#\'{sheet_name}\'!A1", "{fund}")'

    # Fund House Details
    ws_index["C2"] = f"Website: {website}"
    ws_index["C3"] = ""
    ws_index["C4"] = f"Total Schemes: {len(fund_list)}"
    ws_index["C5"] = ""
    ws_index["C6"] = f"Portfolio As On {last_day_previous_month()}"

    # Optional: column widths so it doesn't look tragic
    ws_index.column_dimensions["A"].width = 40
    ws_index.column_dimensions["C"].width = 35

    # ---------------------------
    # 2. FUND SHEETS
    # ---------------------------
    for fund in fund_list:
        sheet_name = fund[:31].replace('"', '')
        ws = wb.create_sheet(title=sheet_name)

        # Top section
        ws["A1"] = '=HYPERLINK("#INDEX!A1", "Go to INDEX")'
        ws["A2"] = fund
        ws["A3"] = "AAUM for the month:"
        ws["A4"] = ""

        ws["A2"].font = Font(bold=True)

        # Table start
        start_row = 5

        # Headers
        ws[f"A{start_row}"] = "Security Name"
        ws[f"B{start_row}"] = "Market Value (In Rupees Million)"
        ws[f"C{start_row}"] = "% of Portfolio Value"

        for col in ["A", "B", "C"]:
            ws[f"{col}{start_row}"].font = Font(bold=True)

        # Placeholder rows
        for i in range(1, 4):
            ws[f"A{start_row + i}"] = ""
            ws[f"B{start_row + i}"] = ""
            ws[f"C{start_row + i}"] = ""
 
        # Section labels (structured properly now)
        # base = start_row + 5

        # ws[f"A{base}"] = "Total"
        # ws[f"A{base+2}"] = "Money Market or Equivalent"
        # ws[f"A{base+4}"] = "Others"
        # ws[f"A{base+6}"] = "Grand Total"

        # # Bold important rows
        # ws[f"A{base}"].font = Font(bold=True)
        # ws[f"A{base+6}"].font = Font(bold=True)

        # Column widths (so humans can read it)
        ws.column_dimensions["A"].width = 25
        ws.column_dimensions["B"].width = 25
        ws.column_dimensions["C"].width = 25

    # ---------------------------
    # SAVE
    # ---------------------------
    wb.save(output_file)
    print(f"Blueprint created: {output_file}")
    

funds = [
    "HDFC Mutual Fund",
    "ICICI Prudential Mutual Fund",
    "SBI Mutual Fund"
]
create_excel_blueprint(funds)

Blueprint created: mutual_funds_blueprint.xlsx


In [6]:
import pandas as pd
import os

from app.parse_table import TableParser


parser = TableParser()

path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Canara Robeco Mutual Fund"

all_data = []

for file in os.listdir(path):
    if not file.endswith(".pdf"):
        continue
    
    pdf_path = os.path.join(path,file)
    
    print(file)
    df = parser.extract_tables_from_pdf(path=pdf_path,pages=None)
    # print(df.shape)
    
    old_cols = list(df.columns)
    
    df["filename"] = file
    
    new_cols = ["filename"] + old_cols
    df = df.reindex(columns = new_cols)
    all_data.append(df)
    
d1 = pd.concat(all_data, axis=0, ignore_index=True)
d1.to_excel("CAN_FINAL.xlsx")

Disclosure-of-Commission 2012-13.pdf
Disclosure-of-Commission 2013-14.pdf
Disclosure-of-Commission 2014-15.pdf
Disclosure-of-Commission 2015-16_.pdf
Disclosure-of-Commission 2016-17.pdf
Disclosure-of-Commission 2017-18.pdf
Disclosure-of-Commission 2018-19.pdf
Disclosure-of-Commission 2019-20.pdf
Disclosure-of-Commission 2020-21.pdf
Disclosure-of-Commission 2021-22.pdf
Disclosure-of-Commission 2022-23.pdf
Disclosure-of-Commission 2023-24.pdf
Disclosure-of-Commission 2024-25.pdf


0